# Mission - Ventes d'une chaîne d'épiceries à Lomé

### Contexte
Une petite chaîne de 4 épiceries à Lomé enregistre chaque vente dans un fichier unique (une ligne = un produit vendu). Les données couvrent le premier semestre 2026 (janvier à juin). Le directeur commercial a remarqué que le mois d'avril semble nettement plus faible que mars.

### Question métier
Pourquoi le chiffre d'affaires d'avril est-il bien plus bas que celui de mars ?

### Données
- Fichier : `ventes_epicerie.csv`
- Colonnes : id_vente, date, boutique, categorie, produit, quantite, prix_unitaire, montant, mode_paiement, code_interne, commentaire

### Objectif
Identifier la ou les causes de la baisse d'avril, en m'appuyant sur les données, et formuler une conclusion claire pour le directeur.

### Plan de travail de ce notebook
`**Nettoyage et contrôle qualité**`

# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Chargement des données

In [2]:
RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW/"ventes_epicerie.csv")

In [3]:
print(f"Dimensions : {df.shape}")
print()
print("Cinq premieres lignes : ")
print()
df.head()

Dimensions : (5080, 11)

Cinq premieres lignes : 



,id_vente,date,boutique,categorie,produit,quantite,prix_unitaire,montant,mode_paiement,code_interne,commentaire
0,V00001,2026-01-01,Tokoin,Conserves,Haricots en boîte,4.0,800,3200,Mobile Money,EXP-2026,NaN
1,V00002,2026-01-01,Tokoin,Hygiène,Dentifrice,2.0,900,1800,Mobile Money,EXP-2026,NaN
2,V00003,2026-01-01,Tokoin,Céréales,Pâtes 500g,1.0,600,600,Espèces,EXP-2026,NaN
3,V00004,2026-01-01,Tokoin,Boissons,Jus de fruit 1L,2.0,1200,2400,Espèces,EXP-2026,NaN
4,V00005,2026-01-01,Tokoin,Hygiène,Savon,1.0,350,350,Espèces,EXP-2026,NaN


In [4]:
# Sauvegarde pour la cerification finale
taille_avant = df.shape

In [5]:
print("Informations : ")
print()
df.info()

Informations : 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5080 entries, 0 to 5079
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_vente       5080 non-null   object 
 1   date           5080 non-null   object 
 2   boutique       5080 non-null   object 
 3   categorie      5080 non-null   object 
 4   produit        5080 non-null   object 
 5   quantite       5055 non-null   float64
 6   prix_unitaire  5080 non-null   int64  
 7   montant        5080 non-null   int64  
 8   mode_paiement  4986 non-null   object 
 9   code_interne   5080 non-null   object 
 10  commentaire    99 non-null     object 
dtypes: float64(1), int64(2), object(8)
memory usage: 436.7+ KB


**Lecture rapide** :
 - 3 variables numériques (quantitatives)
 - 8 variables catégorielles (qualitatives)

# Nettoyage

### 1. Valeurs manquantes

In [6]:
manquantes = pd.DataFrame({
    "nb_manquants" : df.isna().sum(),
    "proportion_%" : (df.isna().mean() * 100).round(2)
})

# Trier par ordre décroissant
manquantes.sort_values("proportion_%", ascending=False, inplace=True)

manquantes

,nb_manquants,proportion_%
commentaire,4981,98.05
mode_paiement,94,1.85
quantite,25,0.49
boutique,0,0.00
date,0,0.00
id_vente,0,0.00
categorie,0,0.00
prix_unitaire,0,0.00
produit,0,0.00
montant,0,0.00


**Lecture rapide** :
 - La colonne `commentaire` est manquante à **98%**

**Action** :
 - Ici nous n'avons pas besoin de la colonne `commentaire`

In [7]:
df.drop(columns=['commentaire'], inplace=True)
print(f"Vérification : {df.columns}")

Vérification : Index(['id_vente', 'date', 'boutique', 'categorie', 'produit', 'quantite',
       'prix_unitaire', 'montant', 'mode_paiement', 'code_interne'],
      dtype='object')


On peut retrouver la quantité en divisant le montant par le prix_unitaire : 

In [8]:
df['quantite'] = (df['montant'] / df['prix_unitaire'])

Le mode de paiement manquant ne genera pas ma resolution. Je vais la remplacer par `Inconnu` :

In [9]:
df['mode_paiement'] = df['mode_paiement'].fillna('Inconnu')

In [10]:
# Verification 
df.isna().sum()

id_vente         0
date             0
boutique         0
categorie        0
produit          0
quantite         0
prix_unitaire    0
montant          0
mode_paiement    0
code_interne     0
dtype: int64

### 2. Doublons

In [11]:
print(f"Nombres de doublons : {df.duplicated().sum()}")

Nombres de doublons : 60


In [12]:
# Suppression des doublons
df.drop_duplicates(inplace=True)

print(f"Nombres de doublons : {df.duplicated().sum()}")

Nombres de doublons : 0


Apres suppression des doublons on n'a plus besoin de la colonne `id_vente`

In [13]:
#Suppression de id_vente
df.drop('id_vente', axis=1, inplace=True)

### 3. Valeurs uniques

In [14]:
for c in df.select_dtypes(include="object").columns :
    print(f"{c}--------------------{df[c].unique()}")
    print()

date--------------------['2026-01-01' '2026-01-02' '2026-01-03' '2026-01-04' '2026-01-05'
 '2026-01-06' '2026-01-07' '2026-01-08' '2026-01-09' '2026-01-10'
 '2026-01-11' '2026-01-12' '2026-01-13' '2026-01-14' '2026-01-15'
 '2026-01-16' '2026-01-17' '2026-01-18' '2026-01-19' '2026-01-20'
 '2026-01-21' '2026-01-22' '2026-01-23' '2026-01-24' '2026-01-25'
 '2026-01-26' '2026-01-27' '2026-01-28' '2026-01-29' '2026-01-30'
 '2026-01-31' '2026-02-01' '2026-02-02' '2026-02-03' '2026-02-04'
 '2026-02-05' '2026-02-06' '2026-02-07' '2026-02-08' '2026-02-09'
 '2026-02-10' '2026-02-11' '2026-02-12' '2026-02-13' '2026-02-14'
 '2026-02-15' '2026-02-16' '2026-02-17' '2026-02-18' '2026-02-19'
 '2026-02-20' '2026-02-21' '2026-02-22' '2026-02-23' '2026-02-24'
 '2026-02-25' '2026-02-26' '2026-02-27' '2026-02-28' '2026-03-01'
 '2026-03-02' '2026-03-03' '2026-03-04' '2026-03-05' '2026-03-06'
 '2026-03-07' '2026-03-08' '2026-03-09' '2026-03-10' '2026-03-11'
 '2026-03-12' '2026-03-13' '2026-03-14' '2026-03-15'

**Action** :
 - Le `code_interne` a une seule valeur unique donc pas d'apport sur mon analyse.

In [15]:
df.drop('code_interne',  axis =1, inplace=True)

### 4. Conversion de la date

In [16]:
df['date'] = pd.to_datetime(
    df['date'],
    format = '%Y-%m-%d',
    errors='coerce'
)

In [17]:
print("Vérification apres conversion : ")
df.isna().sum()

Vérification apres conversion : 


date             0
boutique         0
categorie        0
produit          0
quantite         0
prix_unitaire    0
montant          0
mode_paiement    0
dtype: int64

In [18]:
# Création de la colonne mois
df['mois'] = df['date'].dt.month

# Création de la colonne mois
df['jour'] = df['date'].dt.day

In [19]:
df[['jour','mois']].head()

,jour,mois
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1


# Finalisation de l'analyse

In [20]:
print("Types de données : ")
df.dtypes

Types de données : 


date             datetime64[ns]
boutique                 object
categorie                object
produit                  object
quantite                float64
prix_unitaire             int64
montant                   int64
mode_paiement            object
mois                      int32
jour                      int32
dtype: object

### Condition importante sur les valeurs numériques

In [21]:
condition = (
    (df['prix_unitaire'] > 0) &
    (df['quantite'] > 0) &
    (df['montant'] > 0)
)

df = df.loc[condition]

In [22]:
taille_apres = df.shape

print(f"Dimension initiale : {taille_avant}")
print(f"Dimension finale : {taille_apres}")

Dimension initiale : (5080, 11)
Dimension finale : (5020, 10)


# Sauvegarde

In [23]:
df.to_parquet(CLEAN / "ventes_epicerie_propres", index=False)